In [1]:
import os
import warnings
import polars as pl
import ray
from tqdm import tqdm

warnings.filterwarnings("ignore", message="You are using `torch.load` with `weights_only=False`")

# -------------------------
# Paths / Config
# -------------------------
VIDEO_DIR = "./data/1_video_files"
OCR_DIR = "./data/3_ocr_results"
OUTPUT_DIR = "./data/4_stt_results"

FPS = 10
NUM_WORKERS = 12

ray.init(num_gpus=1)


# -------------------------
# Ray Actor
# -------------------------
@ray.remote(num_gpus=1 / NUM_WORKERS)
class STTWorker:
    def __init__(self):
        import whisper
        self.model = whisper.load_model("turbo")

    def transcribe(self, video_path: str) -> list:
        try:
            result = self.model.transcribe(video_path, language="ko")
            return result.get("segments", [])
        except Exception as e:
            print(f"⚠️ STT 실패: {video_path}, {e}")
            return []


# -------------------------
# Utils
# -------------------------
def get_stt_text_for_timestamp(timestamp: float, segments: list) -> str | None:
    for seg in segments:
        if seg["start"] <= timestamp < seg["end"]:
            return seg["text"].strip() or None
    return None


def add_stt_column(ocr_parquet_path: str, segments: list, fps: int) -> pl.DataFrame:
    """OCR parquet를 읽고 stt_text 컬럼을 추가해서 반환"""
    df = pl.read_parquet(ocr_parquet_path, glob=False)

    stt_texts = []
    for row in df.iter_rows(named=True):
        timestamp = row["frame_num"] / fps
        stt_texts.append(get_stt_text_for_timestamp(timestamp, segments))

    return df.select("frame_num").with_columns(pl.Series("stt_text", stt_texts))


# -------------------------
# Main
# -------------------------
def main():
    channels = sorted([
        d for d in os.listdir(OCR_DIR)
        if os.path.isdir(os.path.join(OCR_DIR, d))
    ])
    print(f"📁 총 {len(channels)}개 채널 발견")

    # 전체 작업 목록 수집: (channel, video_name, ocr_path, video_path, out_path)
    all_jobs = []
    for channel_name in channels:
        ocr_channel_dir = os.path.join(OCR_DIR, channel_name)
        out_channel_dir = os.path.join(OUTPUT_DIR, channel_name)
        video_channel_dir = os.path.join(VIDEO_DIR, channel_name)

        parquet_files = sorted([
            f for f in os.listdir(ocr_channel_dir)
            if f.endswith(".parquet")
        ])

        for pf in parquet_files:
            video_name = pf[:-8]  # .parquet 제거
            out_path = os.path.join(out_channel_dir, pf)

            if os.path.exists(out_path):
                continue  # 이미 완료

            ocr_path = os.path.join(ocr_channel_dir, pf)
            video_path = os.path.join(video_channel_dir, f"{video_name}.mp4")

            if not os.path.exists(video_path):
                print(f"⚠️ 비디오 없음: {video_path}")
                continue

            all_jobs.append((channel_name, video_name, ocr_path, video_path, out_path))

    print(f"🎯 처리 대상: {len(all_jobs)}개 영상")
    if not all_jobs:
        print("✅ 모든 영상 처리 완료!")
        return

    # 워커 생성 + 워밍업
    print(f"🔧 {NUM_WORKERS}개 STT 워커 생성 중...")
    workers = [STTWorker.remote() for _ in range(NUM_WORKERS)]

    print("🔥 워밍업 중...")
    ray.get(workers[0].transcribe.remote(all_jobs[0][3]))
    print("✅ 준비 완료!\n")

    # 태스크 분배
    futures = {}  # future → (ocr_path, out_path)
    for i, (channel, video_name, ocr_path, video_path, out_path) in enumerate(all_jobs):
        worker = workers[i % NUM_WORKERS]
        future = worker.transcribe.remote(video_path)
        futures[future] = (ocr_path, out_path)

    # 결과 수집 → 즉시 저장
    pending = list(futures.keys())
    n_saved = 0

    with tqdm(total=len(pending), desc="STT 처리") as pbar:
        while pending:
            ready, pending = ray.wait(pending, num_returns=min(32, len(pending)), timeout=1.0)
            if not ready:
                continue

            for future in ready:
                segments = ray.get(future)
                ocr_path, out_path = futures[future]

                df = add_stt_column(ocr_path, segments, FPS)
                os.makedirs(os.path.dirname(out_path), exist_ok=True)
                df.write_parquet(out_path)
                n_saved += 1
                pbar.update(1)

    ray.shutdown()
    print(f"\n🎉 전체 완료! {n_saved}개 영상 저장됨")


if __name__ == "__main__":
    main()

2026-03-23 17:25:07,110	INFO worker.py:1786 -- Started a local Ray instance.


📁 총 664개 채널 발견
🎯 처리 대상: 66400개 영상
🔧 12개 STT 워커 생성 중...
🔥 워밍업 중...
✅ 준비 완료!

(raylet) Warning: More than 5000 tasks are pending submission to actor 4c2077715ef311ec708b08fc01000000. To reduce memory usage, wait for these tasks to finish before sending more.


STT 처리: 100%|██████████| 66400/66400 [13:48:17<00:00,  1.34it/s]   



🎉 전체 완료! 66400개 영상 저장됨


In [ ]:
import requests
try:
    r = requests.get("http://localhost:8000/health")
    print(r.status_code, r.text)
except Exception as e:
    print(f"서버 죽음: {e}")

: 